# OOP CA 1 D00255640

## Section 1 - Data Analysis

### Reading the CSV

First I have to read the CSV file. This is done through this code.

In [1]:
import pandas as pd
import csv
mycsv = pd.read_csv('manager_salary_survey.csv')

### Changing the Column Names

The column names for this data are survey questions that are hard to perform analysis on as the names are very long.

In [2]:
new_columns = ['timestamp','age','industry','job','additional job context','salary','additional salary','currency','other currency','additional income context','country','state','city','overall work experience','field work experience','education','gender','race']
mycsv.columns = new_columns

### Basic Information of the Dataset

Here I have gathered the standard information about the dataset, as well as the number of missing stats.

In [3]:
def analyse_csv(csv):
    print('Dataset Info:')
    print(csv.info())

    print('\nMissing Data Stats:')
    print(csv.isnull().sum())

analyse_csv(mycsv)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27989 entries, 0 to 27988
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  27989 non-null  object 
 1   age                        27989 non-null  object 
 2   industry                   27917 non-null  object 
 3   job                        27989 non-null  object 
 4   additional job context     7240 non-null   object 
 5   salary                     27989 non-null  object 
 6   additional salary          20719 non-null  float64
 7   currency                   27989 non-null  object 
 8   other currency             199 non-null    object 
 9   additional income context  3035 non-null   object 
 10  country                    27989 non-null  object 
 11  state                      22993 non-null  object 
 12  city                       27913 non-null  object 
 13  overall work experience    27989

This code calculates the outliers and displays them using the interquartile range. It can be used for any column as long as it has numerical values.

In [4]:
def find_outliers(csv,column_name):
    csv[column_name] = pd.to_numeric(csv[column_name], errors='coerce')
    q1 = csv[column_name].quantile(0.25)
    q3 = csv[column_name].quantile(0.75)
    iqr = q3 - q1
    calculation = (csv[column_name] < (q1-1.5*iqr)) | (csv[column_name]>(q3+1.5*iqr))
    outliers_values= csv[column_name][calculation]
    return outliers_values

find_outliers(mycsv,'salary')

20383       230000.0
20399       200000.0
20408       210000.0
20420       287000.0
20423       200000.0
            ...     
27769       197600.0
27775       198000.0
27800      2000000.0
27857      1800000.0
27902    120000000.0
Name: salary, Length: 328, dtype: float64

### Getting the Unique Values of a Column

This code is encapsulated so I can use it to find unique values of any column if I wish. This means it is very reusable. It shows me the erroneous values that are in the dataset.

In [5]:
def find_unique_values(column_name):
    unique_values = mycsv[column_name].unique()
    return unique_values.tolist()

find_unique_values('country')

['United States',
 'United Kingdom',
 'US',
 'USA',
 'Canada',
 'United Kingdom ',
 'usa',
 'UK',
 'Scotland ',
 'U.S.',
 'United States ',
 'The Netherlands',
 'Australia ',
 'Spain',
 'us',
 'Usa',
 'England',
 'finland',
 'United States of America',
 'France',
 'United states',
 'Scotland',
 'USA ',
 'United states ',
 'Germany',
 'UK ',
 'united states',
 'Ireland',
 'India',
 'Australia',
 'Uk',
 'United States of America ',
 'U.S. ',
 'canada',
 'Canada ',
 'U.S>',
 'ISA',
 'Argentina',
 'Great Britain ',
 'US ',
 'United State',
 'U.S.A',
 'Denmark',
 'U.S.A.',
 'America',
 'Netherlands',
 'netherlands',
 'England ',
 'united states of america',
 'Ireland ',
 'Switzerland',
 'Netherlands ',
 'Bermuda',
 'Us',
 'The United States',
 'United State of America',
 'Germany ',
 'Malaysia',
 'Mexico ',
 'United Stated',
 'South Africa ',
 'Belgium',
 'Northern Ireland',
 'u.s.',
 'South Africa',
 'UNITED STATES',
 'united States',
 'Sweden',
 'Hong Kong',
 'Kuwait',
 'Norway',
 'Sri la

In [6]:
def unique_count(column_name):
    unique_values_count = mycsv[column_name].value_counts()
    return unique_values_count

unique_count('country')

United States     8976
USA               7924
US                2607
Canada            1566
United States      666
                  ... 
U. S                 1
Untied States        1
Virginia             1
United Stattes       1
Rwanda               1
Name: country, Length: 369, dtype: int64

## Section 2 - Data Cleaning

### Converting the Country Names

I first wanted to change the country names of the country column so they would be the correct names instead of the incorrect or misspelled names they were originally. Through this, I got the count of the different countries that were in the country column.

In [7]:
!pip install pycountry

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pycountry: filename=pycountry-22.3.5-py2.py3-none-any.whl size=10681832 sha256=c9102de7df7793921a0f7fb283374027345c079ea2a53968de18a08e2fb5115c
  Stored in directory: /root/.cache/pip/wheels/47/15/92/e6dc85fcb0686c82e1edbcfdf80cfe4808c058813fed0baa8f
Successfully built pycountry

[notice] A new release of pip is available: 23.0.1 -> 23.3.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import pycountry
class CountryValidation:
    def __init__(self, nextcountry=None):
        self.nextcountry=nextcountry
    
    def validate(self, value):
        if self.nextcountry:
            return self.nextcountry.validate(value)
        else:
            return False

class ChangeCountry(CountryValidation):
    def validate(self, value):
        new_names= set(country.name for country in pycountry.countries)
        return value in new_names or super().validate(value)

def validate_country(value, validatecountry):
    return validatecountry.validate(value)

validate_country_instance = ChangeCountry()
column = 'country'

mycsv['is valid'] = mycsv[column].apply(lambda value: validate_country(value, validate_country_instance))
new_country = mycsv[mycsv['is valid']]
new_country['country'].value_counts()

United States          8976
Canada                 1566
United Kingdom          545
Australia               317
Germany                 172
                       ... 
Cayman Islands            1
Trinidad and Tobago       1
Sierra Leone              1
Kuwait                    1
Rwanda                    1
Name: country, Length: 83, dtype: int64

### Converting the Range Columns

Here, I converted the range of the ages, years of overall experience, and years field work experience, of the employees found in the dataset into single values. After doing this, I could find the number of occurrences of each range in the dataset.

In [9]:
class AgeRangeConverter:
    def __init__(self, next_converter=None):
        self.next_converter = next_converter
        return next_converter

class MidpointConverter(AgeRangeConverter):
    def convert(self, age_range):
        try:
            start, end = map(int, age_range.split('-'))
            return (start + end) / 2
        except (ValueError, AttributeError):
            return None

def convert_age_range(df, column_name, converter):
    df['numeric_age'] = df[column_name].apply(converter.convert)
    return df

df = pd.DataFrame(mycsv)
column_to_convert = 'age'

df = convert_age_range(df, column_to_convert, MidpointConverter())

print(f"{column_to_convert} column and number of occurences")
print(df[column_to_convert].value_counts())

age column and number of occurences
25-34         12631
35-44          9879
45-54          3182
18-24          1200
55-64           991
65 or over       94
under 18         12
Name: age, dtype: int64


### Performing Basic Numerical Analysis

In this code I found the mean, median and standard deviation of the age column using the previous code.

In [10]:
print(f"mean ={df['numeric_age'].mean()}")
print(f"median ={df['numeric_age'].median()}")
print(f"standard deviation ={df['numeric_age'].std()}")

mean =36.0258401176344
median =39.5
standard deviation =8.578506871938663


### Converting the Salary and Additional Salary Columns

I found that there was a lot of unrealistic outliers and null values in this dataset, so I created this code to remove the null values and only use the values that I thought were accurate through the use of the interquartile range. Note that the 'data_file' must already be read using 'pd.read_csv'.

In [11]:
class SalaryAnalyser:
    def __init__(self, data_file):
        self.df = data_file

    def salary_clean_analyse(self, column_name):
        self.df = self.df[pd.to_numeric(self.df[column_name], errors='coerce').notna()]
        self.df[column_name] = pd.to_numeric(self.df[column_name], errors='coerce')

        q1 = self.df[column_name].quantile(0.25)
        q3 = self.df[column_name].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        self.df = self.df[(self.df[column_name] >= lower_bound) & (self.df[column_name] <= upper_bound)]

        mean_value = self.df[column_name].mean()
        median_value = self.df[column_name].median()
        std_dev_value = self.df[column_name].std()

        print(f"Mean {column_name} = {mean_value}")
        print(f"Median {column_name} = {median_value}")
        print(f"Standard Deviation {column_name} = {std_dev_value}")

analyser = SalaryAnalyser(mycsv)
analyser.salary_clean_analyse('salary')

Mean salary = 81180.99646691127
Median salary = 75000.0
Standard Deviation salary = 38575.879375270444
/tmp/ipykernel_37/1012846889.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df[column_name] = pd.to_numeric(self.df[column_name], errors='coerce')


### Converting the Education Column

Similarly to the previous code, I can find the Education counts. I noticed there was some null entries in this column so I had to remove them and then could perform the analysis.

In [12]:
class EducationAnalyser:
    def __init__(self, data_file):
        self.df = data_file

    def education_clean_analyse(self, column_name):
        self.df = self.df[self.df[column_name].notna()]

        value_counts = self.df[column_name].value_counts()
        print(f"{column_name} column and number of occurences:\n{value_counts}")

analysis = EducationAnalyser(mycsv)
analysis.education_clean_analyse('education')

education column and number of occurences:
College degree                        13486
Master's degree                        8844
Some college                           2057
PhD                                    1426
Professional degree (MD, JD, etc.)     1322
High School                             638
Name: education, dtype: int64


### Converting the Currency Column

In this code, I changed the 'currency' column so it has removed the 'Nan' and 'Other' values so it is capable of being analysed.

In [13]:
class CurrencyAnalyser:
    def __init__(self, data_file):
        self.df = data_file

    def currency_clean_analyse(self, column_name):
        self.df = self.df[self.df[column_name].notna()]
        self.df = self.df[self.df[column_name] != 'Other']
        
        value_counts = self.df[column_name].value_counts()

        print(f"{column_name} column and number of occurences:\n{value_counts}")

analyser = CurrencyAnalyser(mycsv)
analyser.currency_clean_analyse('currency')

currency column and number of occurences:
USD        23319
CAD         1668
GBP         1587
EUR          639
AUD/NZD      502
CHF           37
SEK           37
JPY           23
ZAR           16
HKD            4
Name: currency, dtype: int64


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=bedead33-6cea-482d-b9b9-f2b85922aeba' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>